In [67]:
#Scraping the Billborad Year End Hot 100 charts from Billboard's website between 1975-2025.
#It is an annual ranking of the 100 most popular songs in the USA each year 

#Importing Libraries:

import requests 
#sends requests to websites 
import pandas as pd 
#stores and helps manipulate data in tables
import time 
#delays between requests so does not overwhelm serves
import os 
#lets us create folders 
from urllib.parse import quote 
#To wrap HTML in
from io import StringIO

print('succesful.')
#checking if all relevant libraries imported succesfully, 


succesful.


In [69]:
#Setting Things up
#Creating Output folder 
os.makedirs('data/raw', exist_ok=True) 

#Defining Year Range of 1975 to 2025
start_year = 1975 
end_year = 2025

#HEADER to make requests look like its coming from a legitimate browser, without this may be blocked
HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

print(f'Scraping Billboard Year End Hot 100 for {start_year}-{end_year}')


Scraping Billboard Year End Hot 100 for 1975-2025


In [79]:
#Scraping
#taking year and returning top 100 songs from that year by web scraping 
#as the billboards chart URL follows a predictable pattern this makes it easier to loop throug the years
# returns a list of dicts with keys: year, position, artist and title, and empty one if not reached

def scraping_wikipedia_year(year): 
    page_title = f'Billboard Year-End Hot 100 singles of {year}'
    url = f'https://en.wikipedia.org/wiki/{quote(page_title)}'  
    
    try:
        #sending GET request 
        response = requests.get(url, headers=HEADERS, timeout=15)
        
        #If request failed due to an error skip this year 
        if response.status_code != 200:
            print(f' Warning: {year} returned status {response.status_code} - skipped')
            return[]

        #reads all HTML tables from the page 
        tables = pd.read_html(StringIO(response.text))

        #First main table that contains over 90 rows year-end ranking, 
        #Ive done this as I ran the code and would fail as it would select the first table it found which wasnt always the one i wanted
        df = None 
        for table in tables: 
            if len(table) >=90:
                df = table.copy()
                break

        if df is None:
            print(f'Warning: No 100 row table found for {year}')
            return[]

        #standardising column names slightly 
        df.columns = [str(col).strip() for col in df.columns]

        #Finding likely rank, title and artist columns 
        rank_col = None 
        title_col = None
        artist_col = None 

        for col in df.columns:
            col_lower = col.lower()
            if rank_col is None and ('no.' in col_lower or 'position' in col_lower or 'rank' in col_lower):
                rank_col = col
            if title_col is None and 'title' in col_lower:
                title_col = col
            if artist_col is None and 'artist' in col_lower:
                artist_col = col 
        if title_col is None or artist_col is None:
            print(f'Warning: Could not identify title/artist columns for {year}')
            print('Columns found:', df.columns.tolist())

        #in case no rank column exists, we create one from row order 
        if rank_col is None:
            df['position'] = range(1, len(df) +1)
            rank_col = 'position'

        #making sure to keep only the first 100 rows 
        df = df[[rank_col, title_col, artist_col]].head(100).copy()
        df.columns = ['position', 'title', 'artist']

        #cleaning values 
        df['year'] = year
        df['position'] = pd.to_numeric(df['position'], errors = 'coerce')
        df['title'] = df['title'].astype(str).str.strip().str.replace('"', "", regex=False)
        df['artist'] = df['artist'].astype(str).str.strip()

        #Droping bad rows
        df = df.dropna(subset=['position', 'title', 'artist'])
        
        results = df[['year', 'position', 'title', 'artist']].to_dict(orient='records')
        
        print(f'{year}: {len(results)} songs scraped')
        return results 

    except Exception as e: 
        print(f'ERROR scraping {year}: {e}')
        return[]


print('scraping function defined')
        


scraping function defined


In [81]:
#The main scraping loop 
#looping through each year and collecting all songs compiling them into a list 
#in case of getting ip temp blocked i added a 2-second delay between requests 

all_songs = []

for year in range(start_year, end_year +1): 
    print(f'Scraping{year}...')
    songs = scraping_wikipedia_year(year)
    all_songs.extend(songs)
    time.sleep(2) #wait 2 seconds between each year
    
print(f'\nTotal songs collected: {len(all_songs)}')


Scraping1975...
1975: 100 songs scraped
Scraping1976...
1976: 100 songs scraped
Scraping1977...
1977: 100 songs scraped
Scraping1978...
1978: 100 songs scraped
Scraping1979...
1979: 100 songs scraped
Scraping1980...
1980: 100 songs scraped
Scraping1981...
1981: 100 songs scraped
Scraping1982...
1982: 100 songs scraped
Scraping1983...
1983: 100 songs scraped
Scraping1984...
1984: 100 songs scraped
Scraping1985...
1985: 100 songs scraped
Scraping1986...
1986: 100 songs scraped
Scraping1987...
1987: 100 songs scraped
Scraping1988...
1988: 100 songs scraped
Scraping1989...
1989: 100 songs scraped
Scraping1990...
1990: 100 songs scraped
Scraping1991...
1991: 100 songs scraped
Scraping1992...
1992: 100 songs scraped
Scraping1993...
1993: 100 songs scraped
Scraping1994...
1994: 100 songs scraped
Scraping1995...
1995: 100 songs scraped
Scraping1996...
1996: 100 songs scraped
Scraping1997...
1997: 100 songs scraped
Scraping1998...
1998: 100 songs scraped
Scraping1999...
1999: 100 songs scraped


#Converting to DataFrame 
songs_df = pd.DataFrame(all_songs)

#Save combined file
output_file = 'data/raw/billboard_year_end_hot_100_1975_2025.csv'
songs_df.to_csv('~/Desktop/billboard_data.csv', index=False)

print(f'Saved to: {output_file}')
print(songs_df.head())